# Hill Cipher Decryption

Decryption inverts each block of ciphertext using the matrix inverse modulo 26:

$$\mathbf{p} \equiv K^{-1} \cdot \mathbf{c} \pmod{26}$$

In [1]:
from typing import List
import sympy
from generate_hill import matrix_mod_inverse


def decrypt(key: List[List[int]], ciphertext: str) -> str:
    matrix = sympy.Matrix(key)
    n = matrix.rows
    inv_key = matrix_mod_inverse(key, 26)
    inv_matrix = sympy.Matrix(inv_key)
    cleaned = "".join(ch.upper() for ch in ciphertext if ch.isalpha())
    if len(cleaned) % n != 0:
        raise ValueError(f"Ciphertext length ({len(cleaned)}) must be a multiple of key size ({n})")
    plaintext_chars = []
    for i in range(0, len(cleaned), n):
        block = [ord(ch) - ord("A") for ch in cleaned[i : i + n]]
        col = sympy.Matrix(block)
        res = (inv_matrix * col) % 26
        plaintext_chars.extend(chr(int(v) + ord("A")) for v in res)
    return "".join(plaintext_chars)

## Round Trip Demonstration

Encrypt and decrypt to verify round-trip accuracy.

In [2]:
from generate_hill import generate_key
from encrypt_hill import encrypt

key = generate_key(2)
message = "HELLO WORLD"
ciphertext = encrypt(key, message)
recovered = decrypt(key, ciphertext)

print("Original Message: ", message)
print("Encrypted Text:   ", ciphertext)
print("Decrypted Text:   ", recovered)

Original Message:  HELLO WORLD
Encrypted Text:    PLIHUIFAKP
Decrypted Text:    HELLOWORLD


## Invertibility Validation Test

Providing a non-invertible matrix (e.g. determinant sharing factors with 26) must raise a `ValueError`.

In [3]:
non_invertible_key = [[2, 4], [4, 8]]
try:
    decrypt(non_invertible_key, ciphertext)
except ValueError as err:
    print("Validation succeeded! Caught expected error:", err)

Validation succeeded! Caught expected error: Matrix is not invertible mod 26: det ≡ 0 (mod 26), gcd(0, 26) = 26 ≠ 1
